# Inventario de datos para la T1 de pavimentos

Este cuaderno descarga SDNET2018 desde Utah State University y documenta únicamente P/CP y P/UP. La unidad es un recorte de imagen; la etiqueta señala fisura visible. No entrena modelos.

**Ruta de uso:** Archivo > Subir cuaderno > seleccionar este archivo. Ejecutar las tres celdas con el triángulo izquierdo, en orden. La primera descarga aproximadamente 528 MB; la segunda inventaría las imágenes; la tercera muestra las evidencias y descarga un ZIP pequeño.

**Fuente:** https://digitalcommons.usu.edu/all_datasets/48/

**Atribución:** Maguire, Dorafshan y Thomas (2018). SDNET2018. Utah State University. CC BY 4.0. https://doi.org/10.15142/T3TD19

Los conteos impresos proceden de esta ejecución. La cantidad publicada por la fuente se conserva por separado en el resumen JSON.

## 1 Descargar o reutilizar el archivo oficial

No es necesario extraerlo ni subirlo a GitHub. Si ya lo descargaste en tu computadora y falla la descarga directa, usa la alternativa indicada en el mensaje de error.

In [ ]:
from pathlib import Path
import urllib.request, zipfile, time

# Si ya subiste SDNET2018.zip al panel Archivos, se reutiliza.
ARCHIVO = Path('/content/SDNET2018.zip')
URL = 'https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=1047&context=all_datasets&filename=2&type=additional'
if not ARCHIVO.exists():
    parcial = ARCHIVO.with_suffix('.zip.part')
    solicitud = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    print('Descargando el ZIP oficial de aproximadamente 528 MB. Mantén esta pestaña abierta.')
    try:
        inicio, recibidos, aviso = time.monotonic(), 0, 0
        with urllib.request.urlopen(solicitud, timeout=90) as respuesta, parcial.open('wb') as destino:
            while True:
                bloque = respuesta.read(1024 * 1024)
                if not bloque:
                    break
                destino.write(bloque)
                recibidos += len(bloque)
                if recibidos - aviso >= 50 * 1024 * 1024:
                    print(f'{recibidos / 1024**2:.0f} MiB recibidos; {time.monotonic()-inicio:.0f} segundos', flush=True)
                    aviso = recibidos
        if not zipfile.is_zipfile(parcial):
            raise ValueError('La respuesta no es un ZIP; puede ser una página de bloqueo.')
        parcial.replace(ARCHIVO)
    except Exception as error:
        if parcial.exists():
            parcial.unlink()
        raise RuntimeError('No se completó la descarga oficial. Abre https://digitalcommons.usu.edu/all_datasets/48/ > Additional Files > SDNET2018.zip. Luego, en Colab: panel Archivos (carpeta izquierda) > Subir al almacenamiento de sesión; selecciona el ZIP descargado y vuelve a ejecutar esta celda. No cambies el nombre SDNET2018.zip.') from error
if not zipfile.is_zipfile(ARCHIVO):
    raise ValueError('SDNET2018.zip no es un ZIP válido. Bórralo del panel Archivos y repite la descarga oficial.')
print('ZIP válido disponible:', ARCHIVO, '| bytes:', ARCHIVO.stat().st_size)


## 2 Crear el inventario con Python

Leerá todas las imágenes de pavimentos, sus dimensiones y huellas de bytes. Registrará errores de lectura. El prefijo antes del guion se conserva como agrupación auxiliar observada: su relación con la fotografía original requiere verificación. La columna de grupo de origen verificado queda vacía.

In [ ]:
from pathlib import Path
import subprocess, sys, importlib.util
if importlib.util.find_spec('PIL') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'Pillow>=10,<13'])
RUTA_SCRIPT = Path('/content/inventariar_sdnet.py')
RUTA_SCRIPT.write_text('"""Inventario reproducible de pavimentos SDNET2018 para la entrega T1.\n\nNo entrena modelos ni modifica las imágenes del archivo de origen.\nUso: python src/inventariar_sdnet.py SDNET2018.zip --salida resultado_t1\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport platform\nimport re\nimport sys\nimport zipfile\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image, ImageDraw, ImageFont, __version__ as pillow_version\n\nSOURCE = "https://digitalcommons.usu.edu/all_datasets/48/"\nREFERENCE_COUNTS = {"con_fisura": 2608, "sin_fisura": 21726, "total": 24334}\nGROUP_NOTE = (\n    "prefijo_archivo se obtiene del nombre antes del primer guion. "\n    "La correspondencia prefijo-fotografia original es una inferencia, no un "\n    "mapeo documentado por el README oficial. Se proponen particiones "\n    "conservadoras por este prefijo, sujetas a verificar la correspondencia. "\n    "No repartir aleatoriamente recortes hermanos entre entrenamiento y prueba."\n)\nFIELDS = [\n    "ruta_zip", "nombre_original", "clase", "etiqueta", "ancho_px", "alto_px",\n    "modo", "canales", "sha256", "prefijo_archivo", "grupo_origen_verificado",\n    "legible", "error",\n]\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef pavement_label(name: str):\n    path = PurePosixPath(name.replace("\\\\", "/"))\n    if path.suffix.lower() not in {".jpg", ".jpeg", ".jpe", ".png"}:\n        return None\n    parts = path.parts\n    if "P" not in parts or path.parent.name not in {"CP", "UP"}:\n        return None\n    return ("con_fisura", 1) if path.parent.name == "CP" else ("sin_fisura", 0)\n\n\ndef font(size: int):\n    for candidate in ["DejaVuSans.ttf", "Arial.ttf"]:\n        try:\n            return ImageFont.truetype(candidate, size)\n        except OSError:\n            pass\n    return ImageFont.load_default()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("archivo_zip", type=Path)\n    parser.add_argument("--salida", type=Path, default=Path("resultado_t1"))\n    args = parser.parse_args()\n    if not args.archivo_zip.is_file():\n        parser.error(f"No existe {args.archivo_zip}. Descargue el ZIP oficial y revise la ruta.")\n    if not zipfile.is_zipfile(args.archivo_zip):\n        parser.error("El archivo no es un ZIP válido. Puede ser una página de error guardada como ZIP.")\n    root = args.salida\n    data = root / "datos"\n    figures = root / "figuras"\n    for directory in [data, figures, root / "docs"]:\n        directory.mkdir(parents=True, exist_ok=True)\n\n    rows, samples = [], {"con_fisura": [], "sin_fisura": []}\n    count_readable, hashes, prefixes = Counter(), Counter(), set()\n    with zipfile.ZipFile(args.archivo_zip) as archive:\n        members = sorted(\n            [member for member in archive.infolist()\n             if not member.is_dir() and pavement_label(member.filename) is not None],\n            key=lambda member: member.filename,\n        )\n        if not members:\n            raise SystemExit("No se encontraron imágenes P/CP o P/UP. Verifique que usa SDNET2018.zip oficial.")\n        for index, member in enumerate(members, start=1):\n            category, label = pavement_label(member.filename)\n            name = PurePosixPath(member.filename.replace("\\\\", "/")).name\n            prefix = name.split("-", 1)[0] if "-" in name else ""\n            if not re.fullmatch(r"\\d+", prefix):\n                prefix = ""\n            prefixes.add(prefix)\n            row = {key: "" for key in FIELDS}\n            row.update(ruta_zip=member.filename, nombre_original=name, clase=category,\n                       etiqueta=label, prefijo_archivo=prefix, legible=0)\n            try:\n                blob = archive.read(member)\n                row["sha256"] = hashlib.sha256(blob).hexdigest()\n                hashes[row["sha256"]] += 1\n                with Image.open(io.BytesIO(blob)) as image:\n                    image.load()\n                    row.update(ancho_px=image.width, alto_px=image.height, modo=image.mode,\n                               canales=len(image.getbands()), legible=1)\n                    count_readable[category] += 1\n                    # One example per observed prefix improves variety without\n                    # asserting that this is a verified original-photo ID.\n                    seen = {entry[0]["prefijo_archivo"] for entry in samples[category]}\n                    if len(samples[category]) < 4 and prefix not in seen:\n                        rgb = image.convert("RGB")\n                        samples[category].append((row.copy(), rgb.copy()))\n                        dest = data / "muestra" / ("C" if label else "U") / name\n                        dest.parent.mkdir(parents=True, exist_ok=True)\n                        dest.write_bytes(blob)\n            except Exception as error:\n                row["error"] = f"{type(error).__name__}: {error}"\n            rows.append(row)\n            if index % 4000 == 0 or index == len(members):\n                print(f"Inventariadas {index:,}/{len(members):,} imágenes", flush=True)\n\n    with (data / "inventario_pavimentos.csv").open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.DictWriter(stream, fieldnames=FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)\n\n    observed = Counter(row["clase"] for row in rows)\n    readable_total = sum(count_readable.values())\n    summary = {\n        "fuente_primaria": SOURCE,\n        "licencia_dataset": "Creative Commons Attribution 4.0 International (CC BY 4.0)",\n        "fecha_ejecucion_utc": datetime.now(timezone.utc).isoformat(),\n        "archivo_origen": args.archivo_zip.name,\n        "bytes_archivo_origen": args.archivo_zip.stat().st_size,\n        "sha256_archivo_origen_calculado": sha256_file(args.archivo_zip),\n        "nota_checksum": "Huella calculada localmente para reproducibilidad; no afirma coincidencia con un checksum publicado.",\n        "conteos_reportados_por_fuente": REFERENCE_COUNTS,\n        "conteos_archivos_en_zip": dict(observed) | {"total": len(rows)},\n        "conteos_imagenes_legibles": dict(count_readable) | {"total": readable_total},\n        "archivos_no_legibles": len(rows) - readable_total,\n        "duplicados_binarios_adicionales": sum(value - 1 for value in hashes.values() if value > 1),\n        "grupos_sha256_repetidos": sum(value > 1 for value in hashes.values()),\n        "prefijos_distintos_observados": len(prefixes - {""}),\n        "advertencia_grupos_origen": GROUP_NOTE,\n        "proporcion_con_fisura_en_archivos": observed["con_fisura"] / len(rows),\n        "dimensiones_observadas": sorted({f\'{r["ancho_px"]}x{r["alto_px"]}\' for r in rows if r["legible"]}),\n        "modos_observados": sorted({r["modo"] for r in rows if r["legible"]}),\n        "entorno": {"python": platform.python_version(), "pillow": pillow_version},\n        "alcance": "Inventario y documentación T1. Sin entrenamiento, EDA ni resultados predictivos.",\n    }\n    (data / "resumen_datos.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")\n\n    # Two rows: four real examples with cracks, then four without cracks.\n    canvas = Image.new("RGB", (1200, 740), "white")\n    draw = ImageDraw.Draw(canvas)\n    for row_index, category in enumerate(["con_fisura", "sin_fisura"]):\n        for col, (record, image) in enumerate(samples[category]):\n            x, y = 20 + col * 295, 16 + row_index * 345\n            image.thumbnail((270, 270))\n            canvas.paste(image, (x, y))\n            draw.text((x, y + 277), f\'{"Con fisura C" if record["etiqueta"] else "Sin fisura U"}\', fill="black", font=font(19))\n            draw.text((x, y + 302), record["nombre_original"], fill="black", font=font(17))\n    draw.text((20, 712), "Fuente SDNET2018 | Maguire, Dorafshan y Thomas (2018) | CC BY 4.0", fill="black", font=font(17))\n    canvas.save(figures / "ejemplos_pavimentos.png")\n    (root / "docs" / "criterio_de_grupos.md").write_text(\n        "# Agrupación y prevención de fuga de información\\n\\n" + GROUP_NOTE + "\\n\\n"\n        "grupo_origen_verificado queda vacío hasta comprobar el mapeo original. "\n        "El prefijo se conserva como identificador auxiliar observado. "\n        "Las variantes de brillo, recortes y aumentos derivados deberán permanecer en el mismo grupo.\\n",\n        encoding="utf-8",\n    )\n    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)\n    print(f"Salida creada en {root.resolve()}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
subprocess.run([sys.executable, str(RUTA_SCRIPT), str(ARCHIVO), '--salida', '/content/resultado_t1'], check=True)


## 3 Ver las evidencias y descargar el resultado

Toma una captura legible del resumen impreso. El ZIP contiene el CSV, JSON, documentación, ocho imágenes originales y un mosaico atribuido. Extrae el ZIP en tu computadora y copia las carpetas en el repositorio. Conserva los archivos completos; no entregues únicamente capturas.

In [ ]:
import json, shutil
from pathlib import Path
from google.colab import files
RAIZ = Path('/content/resultado_t1')
DOCUMENTACION = {'fuente_datos.md': '# Procedencia de los datos\n\nFuente primaria: https://digitalcommons.usu.edu/all_datasets/48/\n\nAutores del dataset: Marc Maguire, Sattar Dorafshan y Robert J. Thomas.\n\nPublicación: 17 de mayo de 2018. Institución: Utah State University.\n\nRuta de descarga: ficha del dataset > Additional Files > SDNET2018.zip.\n\nDescarga directa oficial usada por el cuaderno: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=1047&context=all_datasets&filename=2&type=additional\n\nLa ficha reporta 230 fotografías originales, entre ellas 104 de pavimentos, divididas en recortes de 256 × 256 píxeles y etiquetadas C (con fisura) o U (sin fisura). La unidad analizada es el recorte. No equivale a una losa ni a un tramo completo de carretera.\n\nLa licencia indicada en la ficha es CC BY 4.0. Las imágenes de pavimentos proceden de calles y aceras del campus de USU. Esta procedencia no acredita desempeño en Huancayo ni en otra población local.\n\nConservar el ZIP descargado y el archivo ReadMe_SDNET2018.txt. El inventario calcula una huella SHA256 del archivo concreto utilizado. Una huella calculada localmente registra la versión de bytes utilizada; no equivale a una comprobación contra un checksum oficial.\n\nReferencia del dataset: Maguire, M., Dorafshan, S., & Thomas, R. J. (2018). SDNET2018: A concrete crack image dataset for machine learning applications [Conjunto de datos]. Utah State University. https://doi.org/10.15142/T3TD19\n\nEl artículo de descripción del dataset documenta los conteos por clase del subconjunto P, utilizados como referencia y comparados con el inventario real.\n\nDorafshan, S., Thomas, R. J., & Maguire, M. (2018). SDNET2018: An annotated image dataset for non-contact concrete crack detection using deep convolutional neural networks. Data in Brief, 21, 1664–1668. https://doi.org/10.1016/j.dib.2018.11.015\n\nTexto completo de la publicación original: https://pmc.ncbi.nlm.nih.gov/articles/PMC6247444/\n', 'criterio_de_grupos.md': '# Agrupación y prevención de fuga de información\n\nprefijo_archivo se obtiene del nombre antes del primer guion. La correspondencia prefijo-fotografia original es una inferencia, no un mapeo documentado por el README oficial. Se proponen particiones conservadoras por este prefijo, sujetas a verificar la correspondencia. No repartir aleatoriamente recortes hermanos entre entrenamiento y prueba.\n\ngrupo_origen_verificado queda vacío hasta comprobar el mapeo original. El prefijo se conserva como identificador auxiliar observado. Las variantes de brillo, recortes y aumentos derivados deberán permanecer en el mismo grupo.\n', 'marco_PCS.md': '# Aplicación prevista del marco PCS\n\nReferencia obligatoria: Yu, B., & Barter, R. (2024). Veridical data science. MIT Press. https://vdsbook.com/\n\n## Predictibilidad\n\nEl problema se formula como clasificación binaria de fisura visible por recorte. La anotación C/U permite comparar predicciones con etiquetas existentes. La disponibilidad de etiquetas no garantiza buen desempeño. Se prevé informar sensibilidad de la clase con fisura y precisión, además de la referencia de mayoría. El desequilibrio de clases impide interpretar la exactitud por sí sola. Se reservarán grupos independientes para evaluación; la asignación de prefijos a fotografías originales debe verificarse antes de tratarla como un mapeo confirmado.\n\n## Computabilidad\n\nEl inventario puede ejecutarse en Python leyendo el ZIP directamente. Genera metadatos y huellas SHA256 sin cargar todas las imágenes en memoria. El resumen registra la versión real de Python y Pillow. La fase predictiva se limitará inicialmente a modelos de tamaño compatible con el entorno que el equipo consiga; no se afirma un tiempo de entrenamiento o disponibilidad de GPU que todavía no se haya comprobado.\n\n## Estabilidad\n\nSe propone comparar las predicciones bajo cambios moderados de brillo y distintas particiones por grupo de origen. Las perturbaciones conservarán el contenido físico y la etiqueta visible. Las imágenes transformadas permanecerán en el mismo grupo que su original. Se documentarán las decisiones sobre preprocesamiento, semillas, umbrales y exclusiones. Los casos con respuestas inestables o puntuaciones cercanas al umbral se derivarán a revisión humana. Los umbrales se elegirán con validación y no sobre la prueba reservada.\n\n## Límite de la T1\n\nEstos compromisos describen cómo se evaluará la confiabilidad. T1 presenta el problema, datos, repositorio y justificación PCS; no informa un modelo entrenado ni demuestra aún predictibilidad o estabilidad. Los datos de USU no acreditan transferibilidad automática a pavimentos locales.\n', 'diccionario_datos.md': '# Diccionario del inventario de pavimentos\n\nLos campos siguientes son metadatos generados para documentar el dataset. No representan ensayos mecánicos ni variables originales de diseño. La entrada predictiva futura es la imagen y la variable objetivo es su etiqueta binaria.\n\n| Campo | Tipo | Significado |\n|---|---|---|\n| ruta_zip | Texto | Ruta original del archivo dentro del ZIP oficial. |\n| nombre_original | Texto | Nombre del recorte sin renombrarlo. |\n| clase | Categórico | con_fisura o sin_fisura, según CP o UP. |\n| etiqueta | Entero binario | 1 con fisura visible, 0 sin fisura visible según anotación original. |\n| ancho_px | Entero | Ancho leído realmente, en píxeles. |\n| alto_px | Entero | Alto leído realmente, en píxeles. |\n| modo | Categórico | Modo de imagen leído por Pillow, por ejemplo RGB. |\n| canales | Entero | Número de canales leído realmente. |\n| sha256 | Texto | Huella de bytes del recorte para identificar duplicados exactos. |\n| prefijo_archivo | Texto | Cadena antes del primer guion del nombre. Se conserva como texto para mantener ceros iniciales. |\n| grupo_origen_verificado | Texto | Vacío hasta verificar el mapeo a la fotografía original. |\n| legible | Entero binario | 1 si Pillow pudo abrir y cargar toda la imagen; 0 si falló. |\n| error | Texto | Excepción observada si la imagen no se pudo leer. |\n\nLas imágenes RGB de 256 × 256 contienen 196 608 valores de intensidad por recorte. No se deben anunciar los 13 campos de este inventario como 13 predictores mecánicos. Los píxeles se utilizan como entrada visual; la transformación a características depende del modelo futuro.\n\nEl CSV usa UTF-8 con BOM para facilitar su apertura en Excel. Importar prefijo_archivo como texto, para conservar valores como 001.\n', 'bitacora_ia.md': '# Bitácora de uso de IA\n\nCompletar con la interacción real del equipo. La preparación de este paquete no acredita que sus integrantes hayan ejecutado o verificado el código.\n\n## Registro inicial\n\n- Fecha y hora reales:\n- Integrante responsable:\n- Herramienta y modelo mostrado por la interfaz, si se identifica:\n- Prompt inicial íntegro:\n- Número real de refinamientos posteriores:\n- Finalidad de la ayuda:\n- Archivos o párrafos en los que se utilizó:\n- Fuentes primarias revisadas por el equipo:\n- Prueba efectuada con el código y resultado observado:\n- Error detectado y corrección, o indicar que no se detectó ninguno en esa comprobación:\n- Decisión humana de aceptación, modificación o rechazo:\n\nRepetir el registro para intervenciones adicionales. No inventar cantidad de iteraciones ni comprobaciones.\n\n## Referencia de la herramienta\n\nCompletar según el software efectivamente utilizado y la versión que la interfaz permita identificar. Ejemplo de estructura APA: OpenAI. (s. f.). ChatGPT [Modelo de lenguaje grande]. Recuperado el 27 de octubre de 2026, de https://chatgpt.com/ (Ajustar si la herramienta o su fecha identificable son diferentes.)\n'}
for nombre, texto in DOCUMENTACION.items():
    (RAIZ / 'docs' / nombre).write_text(texto, encoding='utf-8')
resumen = json.loads((RAIZ/'datos/resumen_datos.json').read_text(encoding='utf-8'))
print('Archivos realmente inventariados:', resumen['conteos_archivos_en_zip'])
print('Imágenes legibles:', resumen['conteos_imagenes_legibles'])
print('Archivos no legibles:', resumen['archivos_no_legibles'])
print('Dimensiones:', resumen['dimensiones_observadas'], '| modos:', resumen['modos_observados'])
print('Prefijos observados:', resumen['prefijos_distintos_observados'], '(mapeo a origen pendiente de verificar)')
from IPython.display import display, Image as Imagen
# Guarda una captura de este resumen y revisa nombres y etiquetas de los ejemplos.
display(Imagen(filename=str(RAIZ/'figuras/ejemplos_pavimentos.png'), width=1000))
shutil.make_archive('/content/evidencias_t1', 'zip', RAIZ)
files.download('/content/evidencias_t1.zip')


## Alternativa opcional para un ZIP que ya está en Google Drive

Usa esta alternativa solo si el ZIP ya está en tu Drive. Copia el bloque siguiente en una celda de código situada **antes de la celda 1** y ajusta la ruta a la ubicación real. Después ejecuta las celdas 1, 2 y 3.

```python
from google.colab import drive
from pathlib import Path
import os
drive.mount("/content/drive")
# Ajusta esta ruta al nombre real de tu carpeta.
ORIGEN_DRIVE = Path("/content/drive/MyDrive/SDNET2018.zip")
if not ORIGEN_DRIVE.is_file():
    raise FileNotFoundError(ORIGEN_DRIVE)
DESTINO = Path("/content/SDNET2018.zip")
if not DESTINO.exists():
    os.symlink(ORIGEN_DRIVE, DESTINO)
```

No es necesario habilitar GPU para este inventario. Al cerrar la sesión de Colab se puede perder el almacenamiento temporal: descarga el ZIP de evidencias antes de salir.